# EDA Template

A reusable exploratory data analysis notebook. **Edit only the config cell**, then run top to bottom.

**Sections**
0. Setup and config
1. Dataset overview
2. Data quality (missing values, duplicates)
3. Univariate analysis (numeric, categorical, date)
4. Bivariate and multivariate analysis
5. Outliers
6. Target analysis (optional)
7. Findings and next steps

## 0. Setup and config

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# ---------------- CONFIG: edit this cell only ----------------
FILE_PATH      = "Data/Dataset.csv"   # path to your CSV
TARGET         = None                  # optional target column name, e.g. "churn"
ID_COLS        = []                    # identifier columns to exclude from analysis, e.g. ["show_id"]
DATE_COLS      = []                    # date columns; columns with "date" in the name are also tried
MULTI_VALUE    = {}                    # comma-separated columns, e.g. {"listed_in": ",", "country": ","}
PLACEHOLDERS   = ["nan", "NaN", "None", "N/A", "NA", "Not Given", "", " "]   # text that really means missing
MAX_CATEGORIES = 15                    # at or below this many unique values, a text column is "categorical"
TOP_N          = 10                    # bars shown in top-N charts
SAVE_FIGS      = True
FIG_DIR        = "figures"
# --------------------------------------------------------------

if SAVE_FIGS:
    os.makedirs(FIG_DIR, exist_ok=True)

def savefig(name):
    plt.tight_layout()
    if SAVE_FIGS:
        plt.savefig(f"{FIG_DIR}/{name}.png", dpi=200)
    plt.show()

ModuleNotFoundError: No module named 'seaborn'

### Load data

In [ ]:
df_raw = pd.read_csv(FILE_PATH)
df = df_raw.replace(PLACEHOLDERS, np.nan)      # unify missing markers
df = df.drop(columns=[c for c in ID_COLS if c in df.columns])

# Parse date columns
date_cols = [c for c in df.columns if c in DATE_COLS or "date" in c.lower()]
for c in date_cols:
    df[c] = pd.to_datetime(df[c], errors="coerce")

# Classify columns
numeric_cols = df.select_dtypes(include="number").columns.tolist()
text_cols    = [c for c in df.select_dtypes(include="object").columns if c not in MULTI_VALUE]
cat_cols     = [c for c in text_cols if df[c].nunique() <= MAX_CATEGORIES]
highcard_cols = [c for c in text_cols if c not in cat_cols]

print("Numeric        :", numeric_cols)
print("Categorical    :", cat_cols)
print("High cardinality:", highcard_cols)
print("Multi-valued   :", list(MULTI_VALUE))
print("Date           :", date_cols)

## 1. Dataset overview

In [ ]:
print("Shape (rows, columns):", df_raw.shape)
print("Memory usage: %.2f MB" % (df_raw.memory_usage(deep=True).sum() / 1e6))
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "unique": df_raw.nunique(),
    "example": df_raw.iloc[0],
})

In [ ]:
df.describe(include="all").T

## 2. Data quality

### 2.1 Missing values

Counts include placeholder text (for example `"Not Given"`) converted to missing in the config.

In [ ]:
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "percent": df.isna().mean() * 100,
}).sort_values("percent", ascending=False)
missing = missing[missing["missing"] > 0]
display(missing)

if len(missing):
    missing["percent"].sort_values().plot.barh(figsize=(8, max(3, 0.4 * len(missing))), color="indianred")
    plt.xlabel("% missing")
    plt.title("Missing values by column")
    savefig("missing_values")
else:
    print("No missing values found.")

**Handling strategy** (fill in per column):

| Column | % missing | Strategy | Reason |
|---|---|---|---|
| | | | |

### 2.2 Duplicates and uniqueness

In [ ]:
print("Fully duplicated rows:", df_raw.duplicated().sum())
for c in ID_COLS:
    if c in df_raw.columns:
        print(f"Duplicate values in {c}:", df_raw[c].duplicated().sum())
constant = [c for c in df.columns if df[c].nunique(dropna=True) <= 1]
print("Constant columns:", constant)

## 3. Univariate analysis

### 3.1 Numeric features

In [ ]:
def numeric_summary(s, name):
    s = s.dropna()
    if s.empty:
        return
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())
    print(f"{name}: mean={s.mean():,.2f}  median={s.median():,.2f}  std={s.std():,.2f}  "
          f"skew={s.skew():.2f}  outliers(IQR)={n_out} ({n_out/len(s):.1%})")
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
    sns.histplot(s, kde=True, ax=ax[0])
    sns.boxplot(x=s, ax=ax[1])
    fig.suptitle(name)
    savefig(f"uni_num_{name}")

for c in numeric_cols:
    numeric_summary(df[c], c)
if not numeric_cols:
    print("No numeric columns.")

### 3.2 Categorical features

In [ ]:
def categorical_summary(s, name, top=TOP_N):
    s = s.dropna()
    if s.empty:
        return
    vc = s.value_counts()
    print(f"{name}: unique={s.nunique()}  mode={vc.index[0]} ({vc.iloc[0] / len(s):.1%})")
    plt.figure(figsize=(8, max(3, 0.4 * min(top, len(vc)))))
    sns.barplot(x=vc.head(top).values, y=vc.head(top).index.astype(str), color="steelblue")
    plt.title(f"{name} (top {min(top, len(vc))})")
    plt.xlabel("count")
    savefig(f"uni_cat_{name}")

for c in cat_cols + highcard_cols:
    categorical_summary(df[c], c)

### 3.3 Multi-valued features (comma-separated)

In [ ]:
for c, sep in MULTI_VALUE.items():
    if c in df.columns:
        exploded = df[c].dropna().astype(str).str.split(sep).explode().str.strip()
        print(f"{c}: average values per row = {df[c].dropna().astype(str).str.split(sep).str.len().mean():.2f}")
        categorical_summary(exploded, c)
if not MULTI_VALUE:
    print("No multi-valued columns configured.")

### 3.4 Date features

In [ ]:
for c in date_cols:
    d = df[c].dropna()
    if d.empty:
        continue
    print(f"{c}: {d.min().date()} to {d.max().date()}  | missing: {df[c].isna().sum()}")
    by_year = d.dt.year.value_counts().sort_index()
    plt.figure(figsize=(8, 3.5))
    plt.plot(by_year.index, by_year.values, marker="o")
    plt.title(f"Records per year: {c}")
    plt.xlabel("year"); plt.ylabel("count")
    savefig(f"uni_date_{c}")
if not date_cols:
    print("No date columns.")

## 4. Bivariate and multivariate analysis

### 4.1 Correlation (numeric features)

In [ ]:
if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr()
    plt.figure(figsize=(max(6, len(numeric_cols)), max(5, len(numeric_cols) * 0.8)))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
    plt.title("Correlation matrix (Pearson)")
    savefig("correlation")

    pairs = corr.where(~np.eye(len(corr), dtype=bool)).stack()
    pairs = pairs[~pairs.index.map(lambda p: p[0] > p[1])]      # drop mirrored pairs
    display(pairs.reindex(pairs.abs().sort_values(ascending=False).index).head(10).to_frame("correlation"))
else:
    print("Need at least 2 numeric columns.")

### 4.2 Numeric vs categorical

In [ ]:
low_card = [c for c in cat_cols if 2 <= df[c].nunique() <= 8]
if numeric_cols and low_card:
    cat = low_card[0]                       # change this to any categorical column
    for num in numeric_cols[:4]:
        plt.figure(figsize=(7, 3.5))
        sns.boxplot(data=df, x=cat, y=num)
        plt.title(f"{num} by {cat}")
        savefig(f"bi_{num}_by_{cat}")
        display(df.groupby(cat)[num].agg(["count", "mean", "median"]))
else:
    print("Need a numeric column and a categorical column with 2 to 8 categories.")

### 4.3 Categorical vs categorical

In [ ]:
if len(low_card) >= 2:
    a, b = low_card[0], low_card[1]         # change these to any two categorical columns
    ct = pd.crosstab(df[a], df[b], normalize="index") * 100
    plt.figure(figsize=(8, 4))
    sns.heatmap(ct, annot=True, fmt=".0f", cmap="Blues")
    plt.title(f"{a} vs {b} (row %)")
    savefig(f"bi_{a}_vs_{b}")
else:
    print("Need two categorical columns with 2 to 8 categories.")

## 5. Outliers

In [ ]:
rows = []
for c in numeric_cols:
    s = df[c].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = int(((s < lo) | (s > hi)).sum())
    rows.append({"column": c, "lower_fence": lo, "upper_fence": hi, "outliers": n, "percent": n / len(s) * 100})
display(pd.DataFrame(rows))

**Decision:** outliers are reported, not deleted by default. Remove or cap a value only if it is clearly an error.

## 6. Target analysis (optional)

Set `TARGET` in the config cell to use this section.

In [ ]:
if TARGET and TARGET in df.columns:
    t = df[TARGET]
    if t.dtype == "object" or t.nunique() <= MAX_CATEGORIES:
        print(t.value_counts(normalize=True).mul(100).round(1).astype(str) + " %")
        plt.figure(figsize=(5, 3.5))
        sns.countplot(x=t.astype(str), order=t.value_counts().index, color="steelblue")
        plt.title(f"Target: {TARGET}")
        savefig("target_distribution")
        for c in numeric_cols:
            if c != TARGET:
                print(f"\n{c} by {TARGET}")
                display(df.groupby(TARGET)[c].agg(["count", "mean", "median"]))
    else:
        corr_t = df[numeric_cols].corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
        display(corr_t.to_frame(f"correlation with {TARGET}"))
else:
    print("TARGET not set. Skipping.")

## 7. Findings and next steps

**Dataset:** X rows, Y columns; what one row represents.

**Data quality:** most affected columns, duplicates, placeholders found, and the handling strategy.

**Distributions:** key observations for the main numeric, categorical, and date features.

**Relationships:** strongest correlations and any group differences (correlation is not causation).

**Outliers:** which columns, how many, and the decision.

**Next steps:** cleaning to apply, features to engineer, questions for the data owner.